# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Muhammadfahadkhan-max/Week-01-ML-FlyRank-AI-/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## My Lane as an ML Task

I selected **Lane 1: Ranking Signal Analysis**.

This is a **Ranking** machine learning task because the goal is to rank content pages according to their likelihood of needing optimization. Instead of simply classifying pages, the model helps prioritize which pages should be reviewed first based on multiple ranking signals.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!git clone https://github.com/Muhammadfahadkhan-max/Week-01-ML-FlyRank-AI-.git
import os
os.chdir("Week-01-ML-FlyRank-AI-")
print(os.listdir())


Cloning into 'Week-01-ML-FlyRank-AI-'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 141 (delta 53), reused 105 (delta 35), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 1.85 MiB | 6.00 MiB/s, done.
Resolving deltas: 100% (53/53), done.
['notebooks', 'data', '02_your_first_readable_model.ipynb', 'requirements.txt', 'README.md', 'skills', '.gitignore', 'scripts', 'CLAUDE.md', 'DATA_USE.md', '.git', 'submission', 'work', 'AGENTS.md', 'outputs', '.github', 'LICENSE', 'Copy_of_01_first_look_and_discovery.ipynb', 'docs', 'SETUP.md', 'GUIDE.md']


In [13]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 30000
Columns: 44


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target or Proxy

The target is to identify pages that should be prioritized for optimization based on their ranking performance. The target is based on observed search performance and ranking-related signals in the dataset, making it a practical proxy for deciding which content should be improved first.

In [8]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'sample_data']


In [9]:
from google.colab import files
uploaded = files.upload()  # a picker will pop up, choose your CSV

Saving w02_ml_task_framing.ipynb to w02_ml_task_framing.ipynb


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@K, using K=20. Of the top 20 pages the model ranks as highest-priority for refresh, what fraction have a declining or stable trend (trend_direction in ["down","stable"]) combined with meaningful search demand (search_volume > 0)? This is defensible because a content team can only review a fixed number of pages per cycle — the metric asks whether the pages that fit in that review batch are the ones actually worth reviewing, not whether every page in the dataset is scored perfectly.

In [14]:
# Build a simple priority score: worse position + falling trend + more staleness = higher priority
df["priority_score"] = (
    df["avg_position"].fillna(df["avg_position"].median())
    - df["trend_pct"].fillna(0) / 10
    + df["days_since_last_update"].fillna(0) / 30
)

K = 20
top_k = df.sort_values("priority_score", ascending=False).head(K)
hit_rate = (top_k["trend_direction"].isin(["down", "stable"]) & (top_k["search_volume"] > 0)).mean()
print(f"Precision@{K}: {hit_rate:.2f}")
top_k[["content_id", "avg_position", "trend_pct", "trend_direction", "days_since_last_update"]]

Precision@20: 0.20


,content_id,avg_position,trend_pct,trend_direction,days_since_last_update
24445,content_661e1745db72,245.0,NaN,new,20
19920,content_23f1cc8851a9,184.0,NaN,new,104
26873,content_7275a6a3a8eb,165.5,NaN,new,104
16044,content_71a31b831092,161.0,NaN,new,104
18532,content_42c7c72b8391,145.5,NaN,new,20
27923,content_692fda8c52bd,142.0,NaN,new,104
15639,content_cb6c7d58c0bc,144.5,NaN,new,20
2970,content_3e087a5d8f15,138.8,0.0,stable,20
28214,content_13bbd72aea33,118.0,-100.0,down,104
1534,content_abeb1aa40158,113.5,NaN,new,20


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

Unit of analysis: One row = one content page (content_id), with its search performance signals (impressions, clicks, average ranking position, click-through rate, traffic trend) and metadata (age, freshness tier, content type) as of the extraction date. There are 30,000 pages in this dataset, each with a unique content_id.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Rows:", df.shape[0], "| Unique content_id:", df["content_id"].nunique())
df[["content_id", "content_type", "avg_position", "ctr", "trend_pct", "trend_direction", "days_since_last_update"]].head(10)

Rows: 30000 | Unique content_id: 30000


,content_id,content_type,avg_position,ctr,trend_pct,trend_direction,days_since_last_update
0,content_304f48230142,keyword article,10.6,0.76,-41.4,down,20
1,content_a1fb4e703a9e,keyword article,20.3,0.05,-57.7,down,25
2,content_9aa793d4d895,keyword article,36.5,0.09,-60.9,down,20
3,content_331d6c4de07b,keyword article,6.2,0.49,-13.8,stable,22
4,content_d99b7a2d90ca,keyword article,44.0,0.13,-34.7,down,14
5,content_d4084a4bc775,keyword article,8.5,0.03,-38.9,down,20
6,content_9a34b442b552,keyword article,7.0,0.00,-92.3,down,20
7,content_a63219c6e95a,keyword article,21.2,0.06,0.6,stable,22
8,content_5e6c160719bc,keyword article,46.0,0.09,-58.8,down,20
9,content_c27558df2b0c,keyword article,4.9,0.16,-29.2,down,104


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.